In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
import json
from unicodedata import category

with open("questionnaire_structure.json", "r", encoding="utf-8") as f:
    questionnaire_structure = json.load(f)

question_to_cat = {
    question["id"]: category["id"]
    for category in questionnaire_structure
    for question in category["questions"]
}

cat_to_category = {category["id"]: category["title"] for category in questionnaire_structure}

question_to_fullquestion = {
    question["id"]: question["text"]
    for category in questionnaire_structure
    for question in category["questions"]
}


print(question_to_fullquestion)




In [ ]:
# Change this to the folder that contains all participant folders
ROOT_DIR = Path("/Users/joanna.luberadzka/Projects/2026/chataid-web-pilot-analysis/data")


rows = []

for participant_dir in ROOT_DIR.iterdir():

    if participant_dir.name == "experiment_Joanna_1779782845088":
        continue

    if not participant_dir.is_dir():
        continue

    json_path = participant_dir / "experiment_data.json"

    if not json_path.exists():
        print(f"Skipping {participant_dir.name}: no experiment_data.json found")
        continue

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    participant_info = data.get("participant", {})
    metadata = data.get("metadata", {})

    base_row = {
        "participant_folder": participant_dir.name,
        "exportId": data.get("exportId"),
        "alias": participant_info.get("alias"),
        "age": participant_info.get("age"),
        "gender": participant_info.get("gender"),
        "isNativeSpeaker": participant_info.get("isNativeSpeaker"),
        "hearingStatus": participant_info.get("hearingStatus"),
        "isListeningExpert": participant_info.get("isListeningExpert"),
        "usingHeadphones": participant_info.get("usingHeadphones"),
        "appVersion": metadata.get("appVersion"),
        "timestamp": metadata.get("timestamp"),
    }

    training_scores = data.get("training_questionnaire", {})
    experiment_scores = data.get("experiment_questionnaire", {})

    # Combine all data into a single row and adds category prefixes to the questionnaire scores
    row = {
        **base_row,
        **{
            f"{question_to_cat.get(key, 'unknown')}_{key}": value
            for key, value in training_scores.items()
        },
        **{
            f"{question_to_cat.get(key, 'unknown')}_{key}": value
            for key, value in experiment_scores.items()
        },
    }


    rows.append(row)
    df = pd.DataFrame(rows)

flipped_sentiment_columns = ["cat3_understandEffort", "cat4_stressful", "cat4_noiseDifficulty", "cat4_askRepeat"]
for col in df.columns:
    if col in flipped_sentiment_columns:
        df[col] = 8 - df[col]


# Add a column for SNR condition based on appVersion
def extract_snr(app_version):
    if app_version is None:
        return None
    if "SNR=-5" in app_version:
        return -5
    elif "SNR=5" in app_version:
        return 5
    else:
        return None
    
df["SNR_condition"] = df["appVersion"].apply(extract_snr)

In [ ]:
df.head(10)

In [ ]:
# save to CSV
df.to_csv("all_questionnaire_data.csv", index=False)


In [ ]:
# Plot box plots for each category

import matplotlib.pyplot as plt
import seaborn as sns


import matplotlib.pyplot as plt


def get_counts(df_sub, cols):
    return df_sub[cols].apply(lambda c: c.value_counts(normalize=True)
                              .reindex(range(1, 8), fill_value=0)).T * 100


# RESULTS FOR THE INITIAL QUESTIONNAIRE (SILENT) - combined

categories = ["cat1", "cat2"]

# columns grouped by category, then flattened in order
cols_by_cat = {cat: list(df.filter(regex=cat).columns) for cat in categories}
all_cols = [c for cat in categories for c in cols_by_cat[cat]]

fig, ax = plt.subplots(figsize=(14, 6))
counts = get_counts(df, all_cols)
counts.plot(kind='barh', stacked=True, colormap='RdYlGn',
            ax=ax, legend=False)
ax.set_xlabel('Percentage')
ax.set_yticks(range(len(all_cols)))
ax.set_yticklabels([question_to_fullquestion.get(col.split("_", 1)[1], col)
                    for col in all_cols])
ax.invert_yaxis()

# horizontal divider between cat1 and cat2
n1 = len(cols_by_cat["cat1"])
ax.axhline(y=n1 - 0.5, color="black", linestyle="--", linewidth=1)

# category labels at the right side of each section
ax.text(101, (n1 - 1) / 2,
        cat_to_category.get("cat1", "cat1"),
        rotation=270, va="center", ha="left", fontweight="bold")
ax.text(101, n1 + (len(all_cols) - n1 - 1) / 2,
        cat_to_category.get("cat2", "cat2"),
        rotation=270, va="center", ha="left", fontweight="bold")

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, title='Response',
           bbox_to_anchor=(1.05, 0.9), loc='upper left')
fig.tight_layout()
plt.show()


In [ ]:
# RESULTS FOR THE FINAL QUESTIONNAIRE (NOISY) - combined

# Set the font size for better readability
plt.rcParams.update({'font.size': 14})

snr_conditions = [5, -5]  # left, right
categories = ["cat3"]#, "cat5", "cat6"]

# columns grouped by category, then flattened in order
cols_by_cat = {cat: list(df.filter(regex=cat).columns) for cat in categories}
all_cols = [c for cat in categories for c in cols_by_cat[cat]]

# section sizes -> divider positions and label midpoints
sizes = [len(cols_by_cat[cat]) for cat in categories]
boundaries = [sum(sizes[:i + 1]) for i in range(len(sizes) - 1)]  # between groups
midpoints, running = [], 0
for n in sizes:
    midpoints.append(running + (n - 1) / 2)
    running += n

fig, axes = plt.subplots(1, 2,
                         figsize=(14, max(2, len(all_cols) * 0.4)),
                         sharey=True)

for ax, snr in zip(axes, snr_conditions):
    df_sub = df[df["SNR_condition"] == snr]
    counts = get_counts(df_sub, all_cols)
    counts.plot(kind='barh', stacked=True, colormap='RdYlGn',
                ax=ax, legend=False)
    ax.set_title(f"SNR = {snr}")
    ax.set_xlabel('Percentage')
    ax.set_yticks(range(len(all_cols)))
    ax.set_yticklabels([question_to_fullquestion.get(col.split("_", 1)[1], col)
                        for col in all_cols])
    ax.invert_yaxis()

    # dashed dividers between categories
    for b in boundaries:
        ax.axhline(y=b - 0.5, color="black", linestyle="--", linewidth=1)

# category labels on the right side of the right panel
#for cat, mid in zip(categories, midpoints):
    #axes[1].text(105, mid,
    #             cat_to_category.get(cat, cat),
    #             rotation=20, va="center", ha="left", fontweight="bold")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Response',
           bbox_to_anchor=(1.05, 0.9), loc='upper left')
fig.tight_layout()
plt.show()